# 06 - Does judge v2 agree with a human?

In [0]:
%pip install -q databricks-sdk
dbutils.library.restartPython()

In [0]:
# restartPython() resets sys.path
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repo root:", repo_root)

In [0]:
import csv

from src.evaluate import (EvalConfig, JUDGE_VERSION, agreement, evaluate_run,
                          format_summary, load_run, save_evaluations, summarize)

cfg = EvalConfig()
RUN_ID = "20261031T191325Z-09e6e4" # Variant A, dev split, run graded by hand

print("Judge:  ", cfg.judge_endpoint, "| version", JUDGE_VERSION, "|", cfg.n_votes, "votes")
print("Run:    ", RUN_ID)

## Load the run and the manual grades

In [0]:
rows = load_run(spark, cfg, RUN_ID)

user = spark.sql("SELECT current_user()").collect()[0][0]
manual_path = f"/Workspace/Users/{user}/manual_grading_{RUN_ID}.csv"
with open(manual_path, encoding="utf-8") as fh:
    manual = list(csv.DictReader(fh))

graded = sum(1 for m in manual if m["correctness"] or m["refusal"])
print(f"{len(rows)} answers, {len(manual)} manual rows, {graded} of them graded")

if graded < len(manual):
    print("WARNING: not every row is graded, the comparison below will be partial")

## Grade with v2

In [0]:
def progress(position, total, e):
    verdict = (e["judge_error"] or
               (f"{e['correctness']} / {e['groundedness']}" if e["in_scope"]
                else f"{e['refusal']}, invented_rule={e['invented_rule']}"))
    print(f"[{position}/{total}] {e['question_id']:<10} {verdict}")


evaluations = evaluate_run(spark, cfg, RUN_ID, on_progress=progress)

n_failed = sum(1 for e in evaluations if e["judge_error"])
print(f"\n{len(evaluations)} graded, {n_failed} judge errors")
print()
print(format_summary(summarize(evaluations)))

In [0]:
print("Saved to", save_evaluations(spark, cfg, evaluations))